In [ ]:
#pip install psycopg2-binary pandas
#pip install python-dotenv


In [9]:
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

# Charger .env
load_dotenv()

PGHOST = os.getenv("PGHOST")
PGDATABASE = os.getenv("PGDATABASE")
PGUSER = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")
PGPORT = os.getenv("PGPORT")

# Charger CSV
df = pd.read_csv("../data/processed/final_clean.csv")

# Harmoniser colonnes
df = df.rename(columns={
    "unemployment_rate_": "unemployment_rate"
})

# Nettoyage total
df = df.replace({
    np.nan: None,
    "nan": None,
    "NaN": None,
    "N/A": None,
    "": None,
    " ": None,
    "inf": None,
    "-inf": None
})

# Connexion Neon
conn = psycopg2.connect(
    dbname=PGDATABASE,
    user=PGUSER,
    password=PGPASSWORD,
    host=PGHOST,
    port=PGPORT,
    sslmode="require"
)

cursor = conn.cursor()

# Colonnes dans l'ordre exact de la table
columns = [
    "year",
    "country_name",
    "life_evaluation_3_year_average",
    "lower_whisker",
    "upper_whisker",
    "explained_by_log_gdp_per_capita",
    "explained_by_social_support",
    "explained_by_healthy_life_expectancy",
    "explained_by_freedom_to_make_life_choices",
    "explained_by_generosity",
    "explained_by_perceptions_of_corruption",
    "dystopia_residual",
    "key_iso3_year",
    "inflation_cpi",
    "gdp_current_usd",
    "gdp_per_capita_current_usd",
    "unemployment_rate",
    "interest_rate_real",
    "inflation_gdp_deflator",
    "gdp_growth_annual",
    "current_account_balance_gdp",
    "government_expense_of_gdp",
    "government_revenue_of_gdp",
    "tax_revenue_of_gdp",
    "gross_national_income_usd",
    "public_debt_of_gdp",
    "country_code"
]

# Extraire les valeurs
values = [tuple(row[col] for col in columns) for _, row in df.iterrows()]

# Requête SQL simple (pas besoin de ON CONFLICT)
insert_query = f"""
INSERT INTO world_happiness ({", ".join(columns)})
VALUES %s
"""

execute_values(cursor, insert_query, values)

conn.commit()
cursor.close()
conn.close()

print("Import terminé avec succès !")


Import terminé avec succès !
